# 5.2 Densidad de Corriente y Ley de Ohm

Volviendo a nuestra expresión de corriente e incorporando la velocidad de deriva, obtenemos:

$$I = n \, A \, v_d \, q$$

Definimos la **densidad de corriente** ($\vec{J}$) como la corriente por unidad de área, un vector que apunta en la dirección del flujo de cargas positivas:

$$\vec{J} = \frac{I}{A}\hat{n} = n \, q \, \vec{v}_d$$

De forma general, la corriente macroscópica es el flujo de la densidad de corriente a través de una superficie:

$$I = \int \vec{J} \cdot d\vec{A}$$

## Ley de Ohm microscópica

Sabiendo que $v_d$ es proporcional al campo eléctrico, podemos establecer que la densidad de corriente es directamente proporcional al campo eléctrico aplicado. Esta relación es conocida como la Ley de Ohm microscópica:

$$\vec{J} = \sigma \vec{E} \quad \text{o bien} \quad \vec{J} = \frac{1}{\rho}\vec{E}$$

Donde $\sigma$ es la conductividad del material y $\rho$ es su resistividad ($\rho = 1/\sigma$).


In [4]:
from IPython.display import display, HTML

simulacion_ohm_mic_html = """
<div style="border: 1px solid #e0e0e0; padding: 20px; border-radius: 8px; background-color: #f8f9fa; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial, sans-serif;">
    <h3 style="margin-top:0; color: #2c3e50;">Ley de Ohm Microscópica</h3>
    <p style="font-size: 0.95em; color: #555; margin-bottom: 15px;">Observa un conductor con un estrechamiento central. En estado estacionario, la corriente total (I) es la misma en todas las secciones, pero el área transversal cambia.</p>
    
    <div style="display: flex; gap: 20px; flex-wrap: wrap; margin-bottom: 15px;">
        <div style="flex: 1.5; min-width: 300px;">
            <div style="margin-bottom: 8px;">
                <label style="font-weight: 600; display: inline-block; width: 200px;">Corriente Total (I): <span id="ohm_val_I" style="font-weight: normal;">50</span></label>
                <input type="range" id="ohm_slider_I" min="10" max="100" step="5" value="50" style="width: 35%; vertical-align: middle;">
            </div>
            <div style="margin-bottom: 8px;">
                <label style="font-weight: 600; display: inline-block; width: 200px;">Conductividad (σ): <span id="ohm_val_Sigma" style="font-weight: normal;">Alta</span></label>
                <input type="range" id="ohm_slider_Sigma" min="1" max="5" step="1" value="4" style="width: 35%; vertical-align: middle;">
            </div>
            <div style="margin-bottom: 8px;">
                <label style="font-weight: 600; display: inline-block; width: 200px;">Estrechamiento (Grosor): <span id="ohm_val_A2" style="font-weight: normal;">50%</span></label>
                <input type="range" id="ohm_slider_A2" min="20" max="100" step="10" value="50" style="width: 35%; vertical-align: middle;">
            </div>
        </div>

        <div style="flex: 1; min-width: 250px; font-size: 0.9em; background: #fff; padding: 10px; border-radius: 6px; border: 1px solid #ccc;">
            <div style="margin-bottom: 5px;"><u>Zona Ancha (1)</u> vs <u>Zona Estrecha (2)</u></div>
            <div style="margin-bottom: 5px;"><b>Densidad (J):</b> <span id="ohm_val_J1" style="color: #0d6efd;">1.0</span> | <span id="ohm_val_J2" style="color: #dc3545;">2.0</span> au</div>
            <div style="margin-bottom: 5px;"><b>Campo (E):</b> <span id="ohm_val_E1" style="color: #0d6efd;">0.25</span> | <span id="ohm_val_E2" style="color: #dc3545;">0.50</span> au</div>
            <div style="font-size: 0.8em; color: #777; margin-top: 5px;">* J = I / A <br>* E = J / σ (Ley de Ohm Microscópica)</div>
        </div>
    </div>
    
    <div style="text-align: center; position: relative;">
        <canvas id="ohm_canvas" width="600" height="250" style="background: #ffffff; border: 1px solid #ccc; border-radius: 4px; box-shadow: 0 2px 5px rgba(0,0,0,0.05); max-width: 100%;"></canvas>
    </div>
</div>

<script>
    setTimeout(function() {
        const canvas = document.getElementById('ohm_canvas');
        if (!canvas) return;
        const ctx = canvas.getContext('2d');
        
        const slider_I = document.getElementById('ohm_slider_I');
        const slider_Sigma = document.getElementById('ohm_slider_Sigma');
        const slider_A2 = document.getElementById('ohm_slider_A2');
        
        // Elementos UI
        const val_I = document.getElementById('ohm_val_I');
        const val_Sigma = document.getElementById('ohm_val_Sigma');
        const val_A2 = document.getElementById('ohm_val_A2');
        const val_J1 = document.getElementById('ohm_val_J1');
        const val_J2 = document.getElementById('ohm_val_J2');
        const val_E1 = document.getElementById('ohm_val_E1');
        const val_E2 = document.getElementById('ohm_val_E2');
        
        let electrons = [];
        const num_electrons = 200;
        
        function initElectrons() {
            electrons = [];
            for(let i=0; i<num_electrons; i++) {
                electrons.push({
                    x: Math.random() * canvas.width,
                    y: Math.random() // Posición relativa en Y (0 a 1) para escalar con el tubo
                });
            }
        }

        function drawArrow(x, y, length, color, label) {
            ctx.beginPath();
            ctx.moveTo(x, y);
            ctx.lineTo(x + length, y);
            ctx.strokeStyle = color;
            ctx.lineWidth = 2;
            ctx.stroke();
            
            // Punta de flecha
            ctx.beginPath();
            ctx.moveTo(x + length, y);
            ctx.lineTo(x + length - 6, y - 4);
            ctx.lineTo(x + length - 6, y + 4);
            ctx.fillStyle = color;
            ctx.fill();
            
            if(label) {
                ctx.fillStyle = color;
                ctx.font = "bold 12px Arial";
                ctx.fillText(label, x + length/2 - 5, y - 8);
            }
        }

        function animate() {
            ctx.clearRect(0, 0, canvas.width, canvas.height);
            
            let I = parseFloat(slider_I.value);
            let sigma = parseFloat(slider_Sigma.value);
            let A2_percent = parseFloat(slider_A2.value) / 100;
            
            val_I.innerText = I;
            val_A2.innerText = slider_A2.value + "%";
            let sigma_labels = ["Muy Baja", "Baja", "Media", "Alta", "Muy Alta"];
            val_Sigma.innerText = sigma_labels[sigma - 1];
            
            // Geometría del conductor
            let H1 = 160; // Altura zona ancha
            let H2 = H1 * A2_percent; // Altura zona estrecha
            let cy = canvas.height / 2;
            
            let x_start = 150;
            let x_end = 450;
            let transition = 40;
            
            // Dibujar Conductor (Fondo)
            ctx.fillStyle = "#e9ecef";
            ctx.beginPath();
            ctx.moveTo(0, cy - H1/2);
            ctx.lineTo(x_start, cy - H1/2);
            ctx.lineTo(x_start + transition, cy - H2/2);
            ctx.lineTo(x_end - transition, cy - H2/2);
            ctx.lineTo(x_end, cy - H1/2);
            ctx.lineTo(canvas.width, cy - H1/2);
            
            ctx.lineTo(canvas.width, cy + H1/2);
            ctx.lineTo(x_end, cy + H1/2);
            ctx.lineTo(x_end - transition, cy + H2/2);
            ctx.lineTo(x_start + transition, cy + H2/2);
            ctx.lineTo(x_start, cy + H1/2);
            ctx.lineTo(0, cy + H1/2);
            ctx.closePath();
            ctx.fill();
            ctx.strokeStyle = "#adb5bd";
            ctx.lineWidth = 2;
            ctx.stroke();

            // Cálculos Físicos
            let J1 = I / H1;
            let J2 = I / H2;
            let E1 = J1 / sigma;
            let E2 = J2 / sigma;
            
            val_J1.innerText = (J1 * 10).toFixed(2);
            val_J2.innerText = (J2 * 10).toFixed(2);
            val_E1.innerText = (E1 * 100).toFixed(2);
            val_E2.innerText = (E2 * 100).toFixed(2);

            // Mover y dibujar electrones
            ctx.fillStyle = "#0d6efd";
            electrons.forEach(e => {
                let current_H = H1;
                let current_J = J1;
                
                // Determinar en qué zona está la partícula
                if (e.x > x_start + transition && e.x < x_end - transition) {
                    current_H = H2;
                    current_J = J2;
                } else if (e.x >= x_start && e.x <= x_start + transition) {
                    // Interpolación lineal simple para la transición
                    let ratio = (e.x - x_start) / transition;
                    current_H = H1 - (H1 - H2) * ratio;
                    current_J = I / current_H;
                } else if (e.x >= x_end - transition && e.x <= x_end) {
                    let ratio = (e.x - (x_end - transition)) / transition;
                    current_H = H2 + (H1 - H2) * ratio;
                    current_J = I / current_H;
                }
                
                // La velocidad de deriva es proporcional a J
                let vd = current_J * 2; 
                e.x += vd;
                
                if (e.x > canvas.width) e.x = 0;
                
                // Calcular la posición Y real basada en la altura actual del tubo
                let real_y = cy - (current_H/2) + (e.y * current_H);
                
                ctx.beginPath();
                ctx.arc(e.x, real_y, 2.5, 0, Math.PI * 2);
                ctx.fill();
            });
            
            // Dibujar vectores macroscópicos representativos
            drawArrow(50, cy - 30, J1 * 50, "#0d6efd", "J");
            drawArrow(50, cy + 30, E1 * 500, "#dc3545", "E");
            
            drawArrow(300, cy - 15, J2 * 50, "#0d6efd", "J");
            drawArrow(300, cy + 15, E2 * 500, "#dc3545", "E");
            
            requestAnimationFrame(animate);
        }

        slider_I.addEventListener('input', () => {});
        slider_Sigma.addEventListener('input', () => {});
        slider_A2.addEventListener('input', () => {});

        initElectrons();
        animate();
    }, 150);
</script>
"""
display(HTML(simulacion_ohm_mic_html))


## Ley de Ohm macroscópica

Si aplicamos la relación microscópica a un segmento de cable de longitud $L$ y área transversal $A$, la diferencia de potencial $\Delta V$ entre los extremos es:

$$\Delta V = E L$$

Sustituyendo el campo eléctrico $E = \rho J$ y sabiendo que $J = I/A$:

$$\Delta V = \left(\rho \frac{I}{A}\right) L = \left(\frac{\rho L}{A}\right) I$$

El término entre paréntesis agrupa las propiedades geométricas del material, definiendo la **Resistencia** ($R$):

$$R = \frac{\rho L}{A}$$

Llegando así a la conocida forma macroscópica de la Ley de Ohm:

$$\Delta V = R I$$

> **Nota:** La unidad de la resistencia es el Ohmio, $1 \, \Omega = 1 \text{ V/A}$

## Dependencia de la temperatura

La resistividad de un material no es estrictamente constante, sino que depende de la temperatura. Para rangos moderados, obedece a una relación lineal:

$$\rho(T) = \rho_0(1 + \alpha(T - T_0))$$

Donde $\alpha$ es el coeficiente térmico del material. Consecuentemente, la resistencia también varía:

$$R(T) = R_0(1 + \alpha(T - T_0))$$

En algunos materiales, al descender la temperatura por debajo de un valor crítico $T_c$, la resistividad cae abruptamente a cero, fenómeno conocido como **superconductividad**.

## Ejemplo: Esferas conductoras concéntricas

Consideremos dos esferas conductoras concéntricas de radios $R_1$ (interior) y $R_2$ (exterior) con un material de resistividad $\rho$ entre ellas. 
Si fluye una corriente $I$ con simetría radial, la densidad de corriente es:

$$\vec{J} = \frac{I}{4\pi r^2}\hat{r}$$

Por la ley de Ohm microscópica, el campo eléctrico en la región intermedia es:

$$E = \rho J = \frac{\rho I}{4\pi r^2}$$

La diferencia de potencial $\Delta V$ entre las esferas se calcula integrando el campo eléctrico:

$$\Delta V = -\int_{R_2}^{R_1} \vec{E} \cdot d\vec{r} = -\int_{R_2}^{R_1} \frac{\rho I}{4\pi r^2} dr = \frac{\rho I}{4\pi} \left[ \frac{1}{r} \right]_{R_2}^{R_1}$$

$$\Delta V = \frac{\rho I}{4\pi} \left( \frac{1}{R_1} - \frac{1}{R_2} \right) = \frac{\rho I}{4\pi} \left( \frac{R_2 - R_1}{R_1 R_2} \right)$$

Dado que macroscópicamente $\Delta V = I R$, podemos identificar la resistencia equivalente de este arreglo:

$$R = \frac{\rho}{4\pi} \left( \frac{R_2 - R_1}{R_1 R_2} \right)$$
